In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install alpaca-trade-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.7/757.7 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 7.1 MB/s eta 0:00:00
  Created wheel for msgpack: filename=msgpack-1.0.3-cp311-cp311-linux_x86_64.whl size=15688 sha256=bf45ef7a51862d4193fb985dc8585d47c2bdc5afb930ac773b092b7d76932f8c
  Stored in directory: /root/.cache/pip/wheels/f6/35/da/ed9b26b510235e00e3a3c3bab7bad97b59214729662255ab3d
Successfully built msgpack
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.1.0
    Uninstalling msgpack-1.1.0:
      Successfully uninstalled msgpack-1.1.0
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling 

In [3]:
import alpaca_trade_api as tradeapi
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import numpy as np

In [4]:
API_KEY = "AK1RX6F8W6QX207XPLDF"
SECRET_KEY = "WaPoTTxkQBGzC51LajCdyw8Pl6svbINa9eDu9TMK"
BASE_URL = "https://api.alpaca.markets"

In [5]:
# Inicializa a API Alpaca com as credenciais lidas do arquivo
api = tradeapi.REST(API_KEY, SECRET_KEY, BASE_URL, api_version='v2')

In [6]:
symbol = "AAPL"
timeframe = "5Min"
data_source = "sip"

In [7]:
clock = api.get_clock()

In [8]:
clock

Clock({   'is_open': True,
    'next_close': '2025-04-22T16:00:00-04:00',
    'next_open': '2025-04-23T09:30:00-04:00',
    'timestamp': '2025-04-22T11:53:44.730241131-04:00'})

In [9]:
# Example: UTC timestamps with 'Z'
now   = datetime.now(timezone.utc)
start = (now - timedelta(hours=3) - timedelta(minutes=15)).isoformat()  # ⇒ '2025-04-21T12:49:12.774699+00:00'
end   = (now - timedelta(minutes=15)).isoformat()  # ⇒ '2025-04-21T13:34:12.774699+00:00'

print("Now (UTC):", now)
print("Start (UTC):", start)
print("End   (UTC):", end)

Now (UTC): 2025-04-22 15:53:47.654737+00:00
Start (UTC): 2025-04-22T12:38:47.654737+00:00
End   (UTC): 2025-04-22T15:38:47.654737+00:00


In [10]:
# Fetch the historical data
bars = api.get_bars(
    symbol,
    timeframe,
    start,
    end,
    feed=data_source
).df

In [11]:
bars

,close,high,low,trade_count,open,volume,vwap
timestamp,,,,,,,
2025-04-22 12:40:00+00:00,194.6900,194.6900,194.5000,245,194.6000,7573,194.587820
2025-04-22 12:45:00+00:00,194.8900,194.9100,194.6500,314,194.6900,28560,194.791148
2025-04-22 12:50:00+00:00,195.0900,195.1291,194.8706,307,194.9000,19954,194.947509
2025-04-22 12:55:00+00:00,195.1900,195.2000,195.0001,265,195.1000,13374,195.150332
2025-04-22 13:00:00+00:00,195.0900,195.2000,195.0400,278,195.1500,10692,195.157991
2025-04-22 13:05:00+00:00,195.0000,195.1600,194.9000,347,195.1600,17633,195.049877
2025-04-22 13:10:00+00:00,195.1000,195.1000,194.8319,379,194.9011,87636,194.948749
2025-04-22 13:15:00+00:00,194.9600,195.0999,194.9200,237,195.0999,9636,194.997318
2025-04-22 13:20:00+00:00,195.1400,195.1400,194.7300,471,194.9400,28267,195.006702


In [12]:
data = bars[['vwap', 'trade_count']]

In [13]:
data

,vwap,trade_count
timestamp,,
2025-04-22 12:40:00+00:00,194.587820,245
2025-04-22 12:45:00+00:00,194.791148,314
2025-04-22 12:50:00+00:00,194.947509,307
2025-04-22 12:55:00+00:00,195.150332,265
2025-04-22 13:00:00+00:00,195.157991,278
2025-04-22 13:05:00+00:00,195.049877,347
2025-04-22 13:10:00+00:00,194.948749,379
2025-04-22 13:15:00+00:00,194.997318,237
2025-04-22 13:20:00+00:00,195.006702,471


In [14]:
scaler = joblib.load("/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/minmax_ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min.pkl")

In [34]:
X_VWAP = data[['vwap']].to_numpy()

X_VWAP_scaled = scaler.transform(X_VWAP)

In [35]:
X_VWAP_scaled

array([[0.24718667],
       [0.24766444],
       [0.24803185],
       [0.24850844],
       [0.24852644],
       [0.24827239],
       [0.24803477],
       [0.24814889],
       [0.24817094],
       [0.24997717],
       [0.25174587],
       [0.25192463],
       [0.25302438],
       [0.25423995],
       [0.25365836],
       [0.25406741],
       [0.25536204],
       [0.25657709],
       [0.2565173 ],
       [0.2551092 ],
       [0.25537471],
       [0.25492883],
       [0.25521631],
       [0.25622832],
       [0.25661773],
       [0.25727436],
       [0.25729214],
       [0.25809413],
       [0.25787045],
       [0.25834642],
       [0.25831209],
       [0.25810071],
       [0.25814592],
       [0.25804616],
       [0.25685941],
       [0.25638641]])

In [30]:
X_Trade_Count = data[['trade_count']].to_numpy()

In [31]:
X_Trade_Count

array([[  245],
       [  314],
       [  307],
       [  265],
       [  278],
       [  347],
       [  379],
       [  237],
       [  471],
       [ 1039],
       [26689],
       [10280],
       [ 9144],
       [10091],
       [ 7942],
       [ 7826],
       [10714],
       [ 9325],
       [ 9661],
       [ 7027],
       [ 6371],
       [ 6179],
       [ 5735],
       [ 6988],
       [ 7053],
       [ 8846],
       [ 7702],
       [ 7059],
       [ 5954],
       [ 5423],
       [ 5812],
       [ 8302],
       [ 6297],
       [ 5794],
       [ 5532],
       [ 5972]])

In [19]:
X_combined = np.concatenate([X_VWAP_scaled, X_Trade_Count], axis=1)

In [20]:
X_combined

array([[2.47186671e-01, 2.45000000e+02],
       [2.47664443e-01, 3.14000000e+02],
       [2.48031853e-01, 3.07000000e+02],
       [2.48508439e-01, 2.65000000e+02],
       [2.48526435e-01, 2.78000000e+02],
       [2.48272393e-01, 3.47000000e+02],
       [2.48034767e-01, 3.79000000e+02],
       [2.48148892e-01, 2.37000000e+02],
       [2.48170943e-01, 4.71000000e+02],
       [2.49977172e-01, 1.03900000e+03],
       [2.51745866e-01, 2.66890000e+04],
       [2.51924631e-01, 1.02800000e+04],
       [2.53024375e-01, 9.14400000e+03],
       [2.54239948e-01, 1.00910000e+04],
       [2.53658357e-01, 7.94200000e+03],
       [2.54067408e-01, 7.82600000e+03],
       [2.55362036e-01, 1.07140000e+04],
       [2.56577092e-01, 9.32500000e+03],
       [2.56517302e-01, 9.66100000e+03],
       [2.55109202e-01, 7.02700000e+03],
       [2.55374715e-01, 6.37100000e+03],
       [2.54928829e-01, 6.17900000e+03],
       [2.55216313e-01, 5.73500000e+03],
       [2.56228319e-01, 6.98800000e+03],
       [2.566177

In [21]:
X_Tensor = np.expand_dims(X_combined, axis=0)

In [22]:
X_Tensor

array([[[2.47186671e-01, 2.45000000e+02],
        [2.47664443e-01, 3.14000000e+02],
        [2.48031853e-01, 3.07000000e+02],
        [2.48508439e-01, 2.65000000e+02],
        [2.48526435e-01, 2.78000000e+02],
        [2.48272393e-01, 3.47000000e+02],
        [2.48034767e-01, 3.79000000e+02],
        [2.48148892e-01, 2.37000000e+02],
        [2.48170943e-01, 4.71000000e+02],
        [2.49977172e-01, 1.03900000e+03],
        [2.51745866e-01, 2.66890000e+04],
        [2.51924631e-01, 1.02800000e+04],
        [2.53024375e-01, 9.14400000e+03],
        [2.54239948e-01, 1.00910000e+04],
        [2.53658357e-01, 7.94200000e+03],
        [2.54067408e-01, 7.82600000e+03],
        [2.55362036e-01, 1.07140000e+04],
        [2.56577092e-01, 9.32500000e+03],
        [2.56517302e-01, 9.66100000e+03],
        [2.55109202e-01, 7.02700000e+03],
        [2.55374715e-01, 6.37100000e+03],
        [2.54928829e-01, 6.17900000e+03],
        [2.55216313e-01, 5.73500000e+03],
        [2.56228319e-01, 6.9880000

In [23]:
model = load_model("/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min+fm=vwap+sm=trade_count+tm=+r=36+sort=False+rfm=False+rsm=False+rtm=False+d=+st=minmax+cts=[0]+Lb=True+e=500+es=True+cb=val_accuracy+p=100+bs=128+tl=0.41606152057647705+ta=0.8365758657455444.keras")

In [24]:
predictions = model.predict(X_Tensor)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 653ms/step


In [25]:
predictions

array([[0.29987508]], dtype=float32)

In [26]:
decisive_sensibility = 0.5

predicted_classes = (predictions >= decisive_sensibility).astype(int)

In [27]:
print(predictions)
print(predicted_classes)

[[0.29987508]]
[[0]]
